# Ploting

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
import os
from pathlib import Path

# ==========================================
# 1. FILE SELECTION UI
# ==========================================
print("Opening file selector...")
root = tk.Tk()
root.withdraw()

csv_path = filedialog.askopenfilename(
    title="Select the consolidated spikes CSV file",
    filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
)

if not csv_path:
    print("Operation canceled.")
    exit()

# ==========================================
# 2. CONFIGURATION & PATHS
# ==========================================
input_folder = os.path.dirname(csv_path)
output_folder = os.path.join(input_folder, "Raster_Plots")
Path(output_folder).mkdir(parents=True, exist_ok=True)
recording_id = Path(csv_path).stem

# ==========================================
# 3. LOAD DATA & PLOT
# ==========================================
print(f"\nLoading data from: {os.path.basename(csv_path)}")
df_spikes = pd.read_csv(csv_path)

if df_spikes.empty:
    print("The CSV file is empty. No spikes to plot.")
    exit()

unit_labels = []
unit_spike_times = []

# Group spikes by Neuron_ID
for unit_id, group in df_spikes.groupby("Neuron_ID"):
    unit_labels.append(unit_id)
    unit_spike_times.append(group["Spike_Time_Seconds"].values)

print(f"Found {len(unit_labels)} unique units. Generating raster plot...")

# --- Plotting ---
fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.colormaps.get_cmap('tab20')

ax.eventplot(
    unit_spike_times,
    colors=[colors(i % 20) for i in range(len(unit_spike_times))],
    linewidths=1.0,
    alpha=0.8
)

ax.set_yticks(range(len(unit_labels)))
ax.set_yticklabels(unit_labels, fontsize=9)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Sorted Units", fontsize=12)
ax.set_title(f"Sorted Units Raster Plot - {recording_id}", fontsize=14)
ax.grid(True, axis='x', linestyle='--', alpha=0.5)

# Save the figure
plot_filename = Path(output_folder) / f"{recording_id}_raster.jpg"
plt.tight_layout()
plt.savefig(plot_filename, format='jpg', dpi=150)
plt.close(fig)

print(f"\nSuccess! Saved raster plot to: {plot_filename}")

Opening file selector...

Loading data from: all_spikes_consolidated_test.csv
Found 76 unique units. Generating raster plot...

Success! Saved raster plot to: /home/samuel/Documentos/Explora/spike_sorter/data/MEA36/single_channel_sorting/test/Raster_Plots/all_spikes_consolidated_test_raster.jpg


# Network properties

In [2]:
import numpy as np

def calculate_run_time(spikes, dt, t_start, t_end):
    """Calculate the proportion of total time within +/- dt of any spike."""
    if len(spikes) == 0:
        return 0.0
    
    # Calculate time windows [spike - dt, spike + dt]
    windows = np.zeros((len(spikes), 2))
    windows[:, 0] = np.maximum(spikes - dt, t_start)
    windows[:, 1] = np.minimum(spikes + dt, t_end)
    
    # Merge overlapping windows
    merged_windows = []
    current_start, current_end = windows[0]
    
    for i in range(1, len(windows)):
        start, end = windows[i]
        if start <= current_end:
            current_end = max(current_end, end)
        else:
            merged_windows.append((current_start, current_end))
            current_start = start
            current_end = end
    merged_windows.append((current_start, current_end))
    
    # Calculate total time covered by these windows
    total_time = sum([end - start for start, end in merged_windows])
    
    # Return proportion of total recording time
    return total_time / (t_end - t_start)

def calculate_spikes_in_window(spikes_1, spikes_2, dt):
    """Calculate proportion of spikes_1 that fall within +/- dt of any spike_2."""
    if len(spikes_1) == 0 or len(spikes_2) == 0:
        return 0.0
    
    # Fast search to find the closest spike in array 2 for each spike in array 1
    indices = np.searchsorted(spikes_2, spikes_1)
    
    # Check left and right nearest neighbors
    left_valid = (indices > 0) & (np.abs(spikes_1 - spikes_2[np.minimum(indices - 1, len(spikes_2)-1)]) <= dt)
    right_valid = (indices < len(spikes_2)) & (np.abs(spikes_1 - spikes_2[np.minimum(indices, len(spikes_2)-1)]) <= dt)
    
    # Count how many spikes in 1 are within dt of any spike in 2
    matches = np.sum(left_valid | right_valid)
    
    return matches / len(spikes_1)

def compute_sttc(spikes_A, spikes_B, dt, t_start, t_end):
    """Compute the Spike Time Tiling Coefficient (Cutts & Eglen, 2014)."""
    if len(spikes_A) == 0 or len(spikes_B) == 0:
        return 0.0
        
    T_A = calculate_run_time(spikes_A, dt, t_start, t_end)
    T_B = calculate_run_time(spikes_B, dt, t_start, t_end)
    
    P_A = calculate_spikes_in_window(spikes_A, spikes_B, dt)
    P_B = calculate_spikes_in_window(spikes_B, spikes_A, dt)
    
    # Handle mathematical edge cases where denominator might be 0
    if P_A * T_B == 1.0:
        term1 = 0.0
    else:
        term1 = (P_A - T_B) / (1.0 - P_A * T_B)
        
    if P_B * T_A == 1.0:
        term2 = 0.0
    else:
        term2 = (P_B - T_A) / (1.0 - P_B * T_A)
        
    return 0.5 * (term1 + term2)

# Connectivity maps

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

# --- Helper Functions ---
# (Ensure compute_sttc, calculate_run_time, and calculate_spikes_in_window 
# are defined in your Jupyter Notebook above this cell)

# =========================================================
# 1. FILE SELECTION UI
# =========================================================
print("Opening file selector...")
root = tk.Tk()
root.withdraw()

# Using askopenfilenames allows you to select multiple conditions at once
csv_paths = filedialog.askopenfilenames(
    title="Select the sorted spikes CSV files",
    filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
)

if not csv_paths:
    print("Operation canceled.")
    exit()

csv_files = [Path(p) for p in csv_paths]
print(f"\nFound {len(csv_files)} recordings to process.")

# Create a dictionary using the filename (without extension) as the condition name
files = {file_path.stem.replace("_sorted_spikes", ""): file_path for file_path in csv_files}

# =========================================================
# 2. LOAD DATA (ELECTRODE-LEVEL)
# =========================================================
dt = 0.05  
datasets = {}
neuron_sets = []

for cond, filepath in files.items():
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error loading {filepath.name}: {e}")
        continue
    
    # Safety checks to handle slight variations in CSV column names
    channel_col = 'Primary_Channel' if 'Primary_Channel' in df.columns else 'Electrode_ID'
    time_col = 'Spike_Time_s' if 'Spike_Time_s' in df.columns else 'Spike_Time_Seconds'
    
    # Group by physical electrode
    df['NeuronLabel'] = 'E' + df[channel_col].astype(str)
    
    labels = df['NeuronLabel'].unique()
    spike_trains = {}
    
    for label in labels:
        spikes = df[df['NeuronLabel'] == label][time_col].values
        spike_trains[label] = np.sort(np.unique(spikes)) 
    
    datasets[cond] = {
        'spike_trains': spike_trains,
        't_start': df[time_col].min(),
        't_end': df[time_col].max()
    }
    neuron_sets.append(set(labels))

if len(neuron_sets) > 0:
    # Keep only the electrodes that are active across ALL selected conditions
    common_neurons = list(set.intersection(*neuron_sets))
    common_neurons.sort(key=lambda x: int(x[1:])) 
    print(f"Analyzing {len(common_neurons)} common electrodes across sessions.")
else:
    print("\nNo datasets were successfully loaded.")
    exit()

# =========================================================
# 3. COMPUTE RATE-CORRECTED STTC MATRICES
# =========================================================
sttc_matrices = {}
n_neurons = len(common_neurons)

for cond, data in datasets.items():
    print(f"Calculating STTC matrix for {cond}...")
    matrix = np.zeros((n_neurons, n_neurons))
    
    for i in range(n_neurons):
        for j in range(n_neurons):
            if i == j:
                matrix[i, j] = 1.0 
            else:
                spikes_A = data['spike_trains'][common_neurons[i]]
                spikes_B = data['spike_trains'][common_neurons[j]]
                matrix[i, j] = compute_sttc(spikes_A, spikes_B, dt, data['t_start'], data['t_end'])
                
    sttc_matrices[cond] = pd.DataFrame(matrix, index=common_neurons, columns=common_neurons)

# =========================================================
# 4. DYNAMIC PLOTTING (RAW STTC)
# =========================================================
n_conds = len(sttc_matrices)
fig, axes = plt.subplots(1, n_conds, figsize=(7 * n_conds, 6))

if n_conds == 1:
    axes = [axes]

for ax, (cond, matrix) in zip(axes, sttc_matrices.items()):
    sns.heatmap(
        matrix, cmap='coolwarm', vmin=-1, vmax=1, center=0, 
        xticklabels=True, yticklabels=True, ax=ax, square=True, 
        cbar_kws={"shrink": .8}
    )
    ax.set_title(f'{cond}\nRaw Connectivity ($\Delta t = 50ms$)')
    ax.set_xlabel('Electrode ID')
    ax.set_ylabel('Electrode ID')
    ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()

# =========================================================
# 5. PROPORTIONAL THRESHOLDING & BINARIZATION
# =========================================================
thresholded_matrices = {}
binarized_matrices = {}
target_density = 10 

print(f"\n--- Applying {target_density}% Proportional Thresholding ---")

for cond, matrix in sttc_matrices.items():
    np_mat = matrix.values
    off_diagonal_vals = np_mat[~np.eye(np_mat.shape[0], dtype=bool)]
    
    # Safety Check for small matrices
    if len(off_diagonal_vals) == 0:
        print(f"{cond}: Matrix too small to threshold.")
        continue
    
    percentile_cutoff = 100 - target_density
    threshold = np.percentile(off_diagonal_vals, percentile_cutoff)
    print(f" -> {cond}: Keeping STTC >= {threshold:.4f}")
    
    thresh_mat = matrix.copy()
    thresh_mat[thresh_mat < threshold] = 0.0
    np.fill_diagonal(thresh_mat.values, 0.0) 
    thresholded_matrices[cond] = thresh_mat
    
    bin_mat = (matrix >= threshold).astype(int)
    np.fill_diagonal(bin_mat.values, 0)
    binarized_matrices[cond] = bin_mat

# =========================================================
# 6. DYNAMIC PLOTTING (THRESHOLDED STTC)
# =========================================================
if thresholded_matrices:
    fig, axes = plt.subplots(1, n_conds, figsize=(7 * n_conds, 6))
    if n_conds == 1:
        axes = [axes]

    for ax, (cond, thresh_mat) in zip(axes, thresholded_matrices.items()):
        cmap = sns.color_palette("coolwarm", as_cmap=True)
        cmap.set_under('whitesmoke') 
        
        sns.heatmap(
            thresh_mat, cmap=cmap, vmin=0.001, vmax=1, 
            xticklabels=True, yticklabels=True, ax=ax, square=True, 
            cbar_kws={"shrink": .8}
        )
        
        ax.set_title(f'{cond}\n(Top {target_density}% Edges)')
        ax.set_xlabel('Electrode ID')
        ax.set_ylabel('Electrode ID')
        ax.tick_params(axis='both', labelsize=8)

    plt.tight_layout()
    plt.show()

<>:116: SyntaxWarning: invalid escape sequence '\D'
<>:116: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_159035/2694130449.py:116: SyntaxWarning: invalid escape sequence '\D'
  ax.set_title(f'{cond}\nRaw Connectivity ($\Delta t = 50ms$)')


Opening file selector...

Found 1 recordings to process.


/tmp/ipykernel_159035/2694130449.py:116: SyntaxWarning: invalid escape sequence '\D'
  ax.set_title(f'{cond}\nRaw Connectivity ($\Delta t = 50ms$)')


ValueError: invalid literal for int() with base 10: 'Ch59'